# BÀI TẬP: TITANIC
**Nguồn:** kaggle.com/c/titanic (891 dòng)


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)
df.info()

(891, 15)
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    str    
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked

## A.2. Missing values & Duplicate data

In [3]:
print(df.isnull().sum())
print("Số dòng trùng lặp:", df.duplicated().sum())


survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
Số dòng trùng lặp: 107


## A.3. Invalid values

In [4]:
print(df.describe())

for col in ['sex', 'pclass', 'embarked', 'class', 'who', 'alone']:
    print(col, df[col].unique())

print("Age âm:", (df['age'] < 0).sum())
print("Fare âm:", (df['fare'] < 0).sum())
print("Sibsp âm:", (df['sibsp'] < 0).sum())
print("Parch âm:", (df['parch'] < 0).sum())


         survived      pclass         age       sibsp       parch        fare
count  891.000000  891.000000  714.000000  891.000000  891.000000  891.000000
mean     0.383838    2.308642   29.699118    0.523008    0.381594   32.204208
std      0.486592    0.836071   14.526497    1.102743    0.806057   49.693429
min      0.000000    1.000000    0.420000    0.000000    0.000000    0.000000
25%      0.000000    2.000000   20.125000    0.000000    0.000000    7.910400
50%      0.000000    3.000000   28.000000    0.000000    0.000000   14.454200
75%      1.000000    3.000000   38.000000    1.000000    0.000000   31.000000
max      1.000000    3.000000   80.000000    8.000000    6.000000  512.329200
sex <StringArray>
['male', 'female']
Length: 2, dtype: str
pclass [3 1 2]
embarked <StringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str
class <StringArray>
['Third', 'First', 'Second']
Length: 3, dtype: str
who <StringArray>
['man', 'woman', 'child']
Length: 3, dtype: str
alone [False  True]
A

## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [5]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
df[['sibsp', 'parch', 'family_size']].head()


,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
for col in ['age', 'fare', 'family_size']:
    print(col)
    print(" Mean:", df[col].mean())
    print(" Median:", df[col].median())
    print(" Mode:", df[col].mode()[0])


age
 Mean: 29.69911764705882
 Median: 28.0
 Mode: 24.0
fare
 Mean: 32.204207968574636
 Median: 14.4542
 Mode: 8.05
family_size
 Mean: 1.904601571268238
 Median: 1.0
 Mode: 1


## Group 2 — Dispersion

In [7]:
for col in ['age', 'fare', 'family_size']:
    print(col)
    print(" Std:", df[col].std())
    print(" Variance:", df[col].var())
    print(" Range:", df[col].max() - df[col].min())
    q1, q3 = df[col].quantile([0.25, 0.75])
    print(" IQR:", q3 - q1)


age
 Std: 14.526497332334042
 Variance: 211.01912474630802
 Range: 79.58
 IQR: 17.875
fare
 Std: 49.6934285971809
 Variance: 2469.436845743116
 Range: 512.3292
 IQR: 23.0896
family_size
 Std: 1.6134585413550873
 Variance: 2.603248464671686
 Range: 10
 IQR: 1.0


## Group 3 — Location and Shape

In [8]:
for col in ['age', 'fare', 'family_size']:
    print(col)
    print(" Quartiles:", df[col].quantile([0.25, 0.5, 0.75]).tolist())
    print(" Skewness:", df[col].skew())
    print(" Kurtosis:", df[col].kurt())


age
 Quartiles: [20.125, 28.0, 38.0]
 Skewness: 0.38910778230082704
 Kurtosis: 0.17827415364210353
fare
 Quartiles: [7.9104, 14.4542, 31.0]
 Skewness: 4.787316519674893
 Kurtosis: 33.39814088089868
family_size
 Quartiles: [1.0, 1.0, 2.0]
 Skewness: 2.7274414739308535
 Kurtosis: 9.159665970194615


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [9]:
survival_by_class = df.groupby('pclass')['survived'].mean().sort_values(ascending=False)
print(survival_by_class)

diff = survival_by_class.max() - survival_by_class.min()
print("Chênh lệch (điểm %):", diff * 100)


pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64
Chênh lệch (điểm %): 38.72671041713812


## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [10]:
print("Theo giới tính:")
print(df.groupby('sex')['survived'].mean())

print("\nTheo hạng vé:")
print(df.groupby('pclass')['survived'].mean())

print("\nKết hợp cả hai:")
print(df.groupby(['sex', 'pclass'])['survived'].mean().unstack())


Theo giới tính:
sex
female    0.742038
male      0.188908
Name: survived, dtype: float64

Theo hạng vé:
pclass
1    0.629630
2    0.472826
3    0.242363
Name: survived, dtype: float64

Kết hợp cả hai:
pclass         1         2         3
sex                                 
female  0.968085  0.921053  0.500000
male    0.368852  0.157407  0.135447


## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [11]:
print(df.groupby('survived')['fare'].mean())
print(df.groupby('survived')['fare'].median())

corr = df['fare'].corr(df['survived'])
print("Correlation fare-survived:", corr)


survived
0    22.117887
1    48.395408
Name: fare, dtype: float64
survived
0    10.5
1    26.0
Name: fare, dtype: float64
Correlation fare-survived: 0.25730652238496227


## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [12]:
survival_by_family = df.groupby('family_size')['survived'].mean()
print(survival_by_family)

print("Correlation family_size-survived:", df['family_size'].corr(df['survived']))


family_size
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: survived, dtype: float64
Correlation family_size-survived: 0.016638989282745285


## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [13]:
survival_by_town = df.groupby('embark_town')['survived'].mean().sort_values(ascending=False)
print(survival_by_town)


embark_town
Cherbourg      0.553571
Queenstown     0.389610
Southampton    0.336957
Name: survived, dtype: float64


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

Hạng vé có ảnh hưởng rất mạnh đến khả năng sống sót: hạng 1 có tỷ lệ sống sót 63.0%, cao hơn hạng 3 (24.2%) tới 38.7 điểm phần trăm. Tuy nhiên khi so sánh trực tiếp, giới tính mới là yếu tố quyết định lớn hơn — nữ giới sống sót 74.2% trung bình so với chỉ 18.9% ở nam giới, và mức chênh lệch này giữ nguyên xu hướng ở mọi hạng vé (ví dụ nữ hạng 3 vẫn sống sót 50%, cao hơn cả nam hạng 1 chỉ 36.9%) — cho thấy "phụ nữ và trẻ em trước" là yếu tố chi phối mạnh hơn tiền vé hay hạng ghế. Giá vé cũng có tương quan dương với sống sót (r = 0.26, người sống sót trả trung bình 48.4 so với người không sống sót chỉ 22.1), nhưng vì giá vé gắn chặt với hạng vé nên đây nhiều khả năng là hiệu ứng gián tiếp của hạng vé chứ không phải bản thân giá vé quyết định. Quy mô gia đình có quan hệ phi tuyến với sống sót — gia đình 2-4 người có tỷ lệ sống sót cao nhất (55-72%), trong khi người đi một mình chỉ 30.3% và gia đình quá đông (6 người trở lên) tỷ lệ giảm mạnh về gần 0%, nên tương quan tuyến tính đo được rất yếu (r = 0.017) dù thực tế có một mẫu hình rõ ràng dạng hình chuông. Hành khách lên tàu ở Cherbourg có tỷ lệ sống sót cao nhất (55.4%), vượt hẳn Queenstown (39.0%) và Southampton (33.7%) — điều này có thể liên quan đến việc Cherbourg có tỷ lệ hành khách hạng 1 cao hơn, chứ không hẳn do bản thân cảng lên tàu ảnh hưởng đến sống sót.